# Building a RAG System with LangChain

## Author: Jorge Soto Tripiana

# LLM with RAG on PDF Documents

**General Objective:**

Evaluate the impact of RAG in LLM responses and compare the effect different hyperparameters configurations have on the accuracy of responses generated, using a set of documents as knowledge base.


**Specifically:**

1. **RAG vs. No RAG:**
  Demonstrate the improvement in accuracy and coherence while reducing hallucinations when using RAG, compared to a traditional Zero-Shot approach.


2. **Chunking Analysis:**
  Observe how the sizes of chunks extracted from the documents affects the quality of the output.


3. **Parameter *k* Study (Retrieval):**
   Determine the optimal number of retrieved chunks (*k* = 1, 3, 5) to maximize response quality, balancing the amount of context and avoiding noise.


4. **Model Comparison:**
   Compare the performance of different embedding models towards retrieval.


5. **Evaluation:**
   Implement automatic NLP evaluation metrics (ROUGE, BLEU for generation; Hit Rate for the retriever) to determine the performance of each configuration.

## Setting up dependencies


In [ ]:
!pip install -q --upgrade huggingface_hub
!pip install -qU langchain-docling docling langchain-text-splitters
!pip install -q langchain-huggingface
!pip install -q bitsandbytes # 4bit quant
!pip install -q flash-linear-attention # attention optimizations

## Libraries


In [3]:
import torch
import os
import docling_parse
from google.colab import drive
from langchain_docling.loader import DoclingLoader, ExportType
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from docling.chunking import HybridChunker
from transformers import pipeline, BitsAndBytesConfig, AutoTokenizer

In [ ]:
# gemma requires login
from huggingface_hub import login
login()

# Storing the weights in drive, avoiding redundant downloads. (if ran on cloud)

In [4]:
drive.mount("/content/drive")

MODEL_ID = "Qwen/Qwen3.5-9B"
SAVE_DIR = "/content/drive/MyDrive/Models/Qwen3.5-9B-4bit"

if os.path.exists(f"{SAVE_DIR}/config.json"):
    # load model
    pipe = pipeline(
        task="text-generation",
        model=SAVE_DIR,
        dtype=torch.bfloat16,
        device_map="auto",
    )
else:
    # download & store in drive
    pipe = pipeline(
        task="text-generation",
        model=MODEL_ID,
        dtype=torch.bfloat16,
        device_map="auto",
        model_kwargs={"quantization_config": BitsAndBytesConfig(
            # less vram usage & faster compute (bfloat16)
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)},
    )

    pipe.model.save_pretrained(SAVE_DIR)
    AutoTokenizer.from_pretrained(MODEL_ID).save_pretrained(SAVE_DIR)

# get tokenizer
tokenizer = pipe.tokenizer

Mounted at /content/drive


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

## PDFs loader

In [5]:
DOCUMENTS_PATH = [
    "f1_2025_driver_stats.pdf",
    "nurburgring_data.pdf",
    "porsche_gt3rs_992.pdf"
]
# meter headers
loader = DoclingLoader(
    file_path=DOCUMENTS_PATH,
    export_type=ExportType.MARKDOWN
    )

In [ ]:
# load
documents = loader.load()

In [7]:
for d in documents:
  print(d.page_content)

## 2025 DRIVER CHAMPIONSHIP

Final standings and round-by-round points

## FINAL DRIVER STANDINGS

POINTS GAP TO CHAMPION

|   POS | DRIVER        |   POINTS |   BEHIND P1 |
|-------|---------------|----------|-------------|
|     1 | L. Norris     |      423 |           0 |
|     2 | M. Verstappen |      421 |          +2 |
|     3 | O. Piastri    |      410 |         +13 |
|     4 | G. Russell    |      319 |        +104 |
|     5 | C. Leclerc    |      242 |        +181 |
|     6 | L. Hamilton   |      156 |        +267 |
|     7 | K. Antonelli  |      150 |        +273 |
|     8 | A. Albon      |       73 |        +350 |
|     9 | C. Sainz      |       64 |        +359 |
|    10 | F. Alonso     |       56 |        +367 |
|    11 | N. Hulkenberg |       51 |        +372 |
|    12 | I. Hadjar     |       51 |        +372 |
|    13 | O. Bearman    |       41 |        +382 |
|    14 | L. Lawson     |       38 |        +385 |
|    15 | E. Ocon       |       38 |        +385 |
|    16 | 

## Splitting Text

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)

texts = [text_splitter.split_text(d.page_content) for d in documents]

## Inspecting sample splits

In [18]:
# first 5 chunks of each doc
for i, doc in enumerate(texts):
  print(f"Doc {i+1}:")
  for chunk in doc[:5]:
    print('*'*100)
    print(chunk)

Doc 1:
****************************************************************************************************
## 2025 DRIVER CHAMPIONSHIP

Final standings and round-by-round points

## FINAL DRIVER STANDINGS

POINTS GAP TO CHAMPION
****************************************************************************************************
|   POS | DRIVER        |   POINTS |   BEHIND P1 |
|-------|---------------|----------|-------------|
|     1 | L. Norris     |      423 |           0 |
|     2 | M. Verstappen |      421 |          +2 |
|     3 | O. Piastri    |      410 |         +13 |
|     4 | G. Russell    |      319 |        +104 |
|     5 | C. Leclerc    |      242 |        +181 |
|     6 | L. Hamilton   |      156 |        +267 |
|     7 | K. Antonelli  |      150 |        +273 |
****************************************************************************************************
|     8 | A. Albon      |       73 |        +350 |
|     9 | C. Sainz      |       64 |        +359 |
|    10

In [ ]:


# test embedding models Qwen/Qwen3-Embedding-0.6B
# vs google/embeddinggemma-300m

embedding = HuggingFaceEmbeddings("Qwen/Qwen3-Embedding-0.6B")

## LLM Set-Up


In [34]:
messages = [ # add role : system (system prompt)
    {"role" : "system", "content" : "You are a helpful assistant. Answer the user's question using ONLY the provided context. Be extremely concise and direct. Do not include filler words. If the answer is not in the context, simply say 'I don't know.'"},
    {"role" : "user", "content" : "What's the use case for Retrieval Augmented Generation sytems?"},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
out = pipe(
    prompt,
    return_full_text=False,
    max_new_tokens=64, # concise answers
    # Qwen's recommended sampling parameters for non thinking tasks
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0
)
print(out[0]["generated_text"])




**Retrieval-Augmented Generation (RAG)** is an architecture that combines the power of Large Language Models (LLMs) with external knowledge sources. Its primary use case is to solve the fundamental
